# Provenance & License

**Source file:** example_environment.py
**Upstream project:** [Gym-Trading-Env](https://github.com/CLementPerroud/Gym-Trading-Env/)
**Original author:** CLementPerroud
**License:** MIT License (see upstream repository)

**Adapted by: Jason Zheng**

**Changes:**
- Imported environment into pipeline path directory
- 

**Link to source:**
https://github.com/ClementPerroud/Gym-Trading-Env/blob/main/examples/example_environnement.py

In [2]:
import sys
sys.path.append("./src")

import pandas as pd
import numpy as np
import time
from gym_trading_env.environments import TradingEnv
import gymnasium as gym

In [3]:
# Import your datas
df = pd.read_csv('../data/train_data.csv', parse_dates=["date"], index_col="date")
df.sort_index(inplace= True)
df.dropna(inplace= True)
df.drop_duplicates(inplace= True)

# Generating features
#WARNING: the column names need to contain keyword 'feature' !
df["feature_close"] = df["close"].pct_change()
df["feature_open"] = df["open"]/df["close"]
df["feature_high"] = df["high"]/df["close"]
df["feature_low"] = df["low"]/df["close"]
df["feature_volume"] = df["volume"] / df["volume"].rolling(7*24).max()
df.dropna(inplace= True)

In [4]:
# Create your own reward function with the history object
def reward_function(history):
   return np.log(history["portfolio_valuation", -1] / history["portfolio_valuation", -2]) # log(p_t / p_t-1)

env = gym.make(
   "TradingEnv",
   name="BTCUSD",
   df = df,
   windows = 5,
   positions = [ -1, -0.5, 0, 0.5, 1, 1.5, 2], # From -1 (=SHORT), to +1 (=LONG)
   initial_position = 'random', # Initial position
   trading_fees = 0.01/100, # 0.01% per stock buy / sell
   borrow_interest_rate = 0.0003 / 100, # per timestep (= 1h here)
   reward_function = reward_function,
   portfolio_initial_value = 1000, # in FIAT (here, USD)
   max_episode_duration = 500,
   disable_env_checker = True
)

env.add_metric('Position Changes', lambda history : np.sum(np.diff(history['position']) != 0))
env.add_metric('Episode Length', lambda history : len(history['position']))

done, truncated = False, False

observation, info = env.reset()
print(info)

while not done and not truncated:
   action = env.action_space.sample()
   observation, reward, done, truncated, info = env.step(action)
   print(observation)
# Save for render
env.save_for_render()

{'idx': 4, 'step': 0, 'date': np.datetime64('2009-01-12T00:00:00.000000000'), 'position_index': 5, 'position': np.float64(1.5), 'real_position': np.float64(1.5), 'data_turbulence': 0.0, 'data_open': 33.213508487988044, 'data_cci_30': -110.45042662357493, 'data_tic': 'BA', 'data_close_30_sma': 33.803937094552175, 'data_macd': -0.0958306014054883, 'data_boll_lb': 32.41980377905534, 'data_boll_ub': 35.18807041004901, 'data_close_60_sma': 33.803937094552175, 'data_vix': 45.84000015258789, 'data_close': 32.808467864990234, 'data_high': 33.341023449197294, 'data_low': 32.39592198577674, 'data_rsi_30': 27.53812392830926, 'data_volume': 4989200.0, 'data_dx_30': 24.05589583417424, 'data_day': 0.0, 'data_Unnamed: 0': 6, 'portfolio_valuation': 1000.0, 'portfolio_distribution_asset': np.float64(45.71990396420319), 'portfolio_distribution_fiat': 0, 'portfolio_distribution_borrowed_asset': 0, 'portfolio_distribution_borrowed_fiat': np.float64(500.0), 'portfolio_distribution_interest_asset': 0, 'port